## FPL Analysis

Scores and ranks all Premier League players for the upcoming gameweek.

Run all cells top to bottom. Sections:
1. **Config** — constants and scoring weights
2. **Data fetch** — live bootstrap API call
3. **Season weights** — blend function for ppg/form
4. **Load data** — read Excel snapshots
5. **Computations** — cleaning, FDR, normalization
6. **Scoring** — weighted score per position
7. **Recommendations** — top 15 per position
8. **Differentials** — low-ownership picks
9. **Watchlist** — track specific players

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import HTML

plt.style.use('dark_background')

### Config

Scoring weights, column lists, and constants. Edit weights here and re-run cells 6–9 to see updated rankings.

> Position weights sum to **0.80**. The remaining 0.20 comes from the season-aware consistency blend (ppg_last + ppg_current + form), merged at scoring time.

In [ ]:
# Constants / Configs
POSITION_MASTER = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}
PLAYERS_NUMERIC_COLUMNS = ['form', 'points_per_game', 'ep_next', 'influence', 'creativity', 'threat', 'ict_index', 
                           'value_form', 'value_season', 'selected_by_percent', 'expected_goals', 'expected_assists', 
                           'expected_goal_involvements', 'expected_goals_conceded', 'clean_sheets_per_90', 'saves_per_90', 'ppg_last',
                            'defensive_contribution']
PLAYERS_NORMALIZATION_COLUMNS = ['form', 'points_per_game', 'ep_next', 'fdr', 'ict_index', 'chance_of_playing_next_round', 
                                 'clean_sheets_per_90', 'saves_per_90', 'threat', 'ppg_last', 'defensive_contribution']

# Position-specific scoring weights.
# NOTE: Each dict intentionally sums to 0.80, not 1.0.
# The remaining 0.20 is reserved for the season-aware consistency signal
# (ppg_last + ppg_current + form), which is blended separately in FORM_AND_PPG_WEIGHTS
# and merged in at scoring time. Together they sum to 1.0.
#
# Weights are backed by per-position correlation analysis (explore.py).
# Run `python explore.py --save` to regenerate heatmaps and retune if needed.
#
# Columns dampened by minutes_confidence (current season) before normalization:
# clean_sheets_per_90, saves_per_90, defensive_contribution, points_per_game, form
# ppg_last is dampened separately by ppg_last_confidence using last season's minutes (GW0).

GKP_WEIGHTS = {
    'ep_next': 0.15,
    'fdr': 0.15,
    'chance_of_playing_next_round': 0.10,
    'clean_sheets_per_90': 0.275,   # strong signal (0.63 corr) — dampened by minutes_confidence before normalization
    'saves_per_90': 0.125,           # inverted + dampened — GKs on weak teams face more shots but concede more
}

DEF_WEIGHTS = {
    'ep_next': 0.15,
    'fdr': 0.15,
    'chance_of_playing_next_round': 0.05,
    'ict_index': 0.20,               # dominant DEF signal (0.88 corr)
    'defensive_contribution': 0.10,  # combined defensive metric (0.79 corr) — dampened by minutes_confidence before normalization
    'clean_sheets_per_90': 0.15,     # moderate signal (0.36 corr) — dampened by minutes_confidence
}

MID_WEIGHTS = {
    'ep_next': 0.07,
    'fdr': 0.18,
    'chance_of_playing_next_round': 0.10,
    'ict_index': 0.32,               # dominant MID signal (0.92 corr)
    'defensive_contribution': 0.13   # strong secondary signal (0.84 corr) — dampened by minutes_confidence before normalization
}

FWD_WEIGHTS = {
    'ep_next': 0.225,
    'fdr': 0.1,
    'chance_of_playing_next_round': 0.10,
    'threat': 0.375,                 # strongest FWD signal (0.95 corr) — forward-looking, not historical
}
PRESEASON_DATA_SHEET = 'GW0'


### Fetch Live Data

Calls the FPL bootstrap-static API to detect the current gameweek.

In [ ]:
# Get Next Gameweek
bootstrap_response = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/')
bootstrap_response.raise_for_status()
bootstrap_data = bootstrap_response.json()
events = bootstrap_data["events"]

next_gw_title: str | None = None
next_gw_id: int = -1
curr_gw_id: int = 0

for event in events:
  if event["is_next"]:
    next_gw_id = event["id"]
    curr_gw_id = next_gw_id - 1
    next_gw_title = event["name"]
    break

if next_gw_id == -1:
  raise RuntimeError('No upcoming gameweek found.')

# The upcoming GW we are predicting for
# Sheet was fetched before this GW was played
GAMEWEEK = f'GW{curr_gw_id}'

### Season-Aware Weight Blend

Shifts trust from last-season data toward current-season form as the season progresses.

| GWs played | ppg_last | ppg_current | form |
|---|---|---|---|
| 0 (pre-season) | 100% | 0% | 0% |
| 1–3 | 70% | 20% | 10% |
| 4–6 | 40% | 30% | 30% |
| 7–10 | 10% | 40% | 50% |
| 11+ | 0% | 45% | 55% |

In [ ]:
def get_season_weights(gws_played: int) -> tuple[float, float, float]:
  """Returns a tuple indicating the weight distribution of last season's points per game, and current season's points per game and 
  and form.
    
    (last_season_ppg, current_season_ppg, current_season_form)
    """
  if gws_played < 1:
    return 1, 0, 0
  elif gws_played < 4:
    return 0.7, 0.2, 0.1
  elif gws_played < 7:
    return 0.4, 0.3, 0.3
  elif gws_played < 11:
    return 0.1, 0.4, 0.5
  else:
    return 0, 0.45, 0.55

### Load Data

Reads the latest GW sheet from all three Excel files.

In [ ]:
# Load Data.
players_df = pd.read_excel('./data/players_master.xlsx', sheet_name=GAMEWEEK)
players_df_last_season = pd.read_excel('./data/players_master.xlsx', sheet_name=PRESEASON_DATA_SHEET)
teams_df = pd.read_excel('./data/teams_master.xlsx', sheet_name=GAMEWEEK)
fixtures_df = pd.read_excel('./data/fixtures_master.xlsx', sheet_name=GAMEWEEK)

### Computations

Cleaning, type casting, FDR calculation, `minutes_confidence` dampening, and min-max normalization.

In [ ]:
# Computations

# Join last season's ppg from GW0 (pre-season snapshot) using 'code' as the stable cross-season ID.
# 'id' can change between seasons; 'code' is permanent.
# New players with no GW0 entry (e.g. promoted clubs, transfers from abroad) get ppg_last = 0.
players_df_last_season = players_df_last_season[['points_per_game', 'minutes', 'code']].rename(columns={'points_per_game': 'ppg_last', 'minutes': 'minutes_last'})
players_df = players_df.merge(players_df_last_season, on='code', how='left')
players_df['ppg_last'] = players_df['ppg_last'].fillna(0)
players_df['minutes_last'] = players_df['minutes_last'].fillna(0)

players_df['full_name'] = players_df['first_name'] + ' ' + players_df['second_name']
players_df['position'] = players_df['element_type'].map(POSITION_MASTER)
players_df['team_name'] = players_df['team'].map(dict(zip(teams_df['id'], teams_df['name'])))

# FPL returns some numeric columns as strings (e.g. form, ep_next, ict_index).
# Cast them to float before any numeric operation; coerce invalid values to NaN.
for col in PLAYERS_NUMERIC_COLUMNS:
  players_df[col] = pd.to_numeric(players_df[col], errors='coerce')

# chance_of_playing_next_round is null (not 0) when a player has no injury concern.
# Treat null as 100% available.
players_df['chance_of_playing_next_round'] = players_df['chance_of_playing_next_round'].fillna(100)
# ep_next is null pre-season (no GW played yet). Treat as 0 expected points.
players_df['ep_next'] = players_df['ep_next'].fillna(0)

# Keep only players with status a (available), i (injured), d (doubtful).
# Drop s (suspended) and u (unavailable/left club) — not selectable.
players_df = players_df[players_df['status'].isin(['a', 'i', 'd'])]

# minutes_confidence: dampens per-90 and contribution stats for players with few appearances.
# Formula: minutes_played / (gws_played * 90), capped at 1.0.
# Pre-season (curr_gw_id = 0): use full season (38 * 90) as denominator to avoid division by zero.
# A player who played every minute gets 1.0 (full trust); 1-game player gets ~0.026 pre-season.
conf_denominator = curr_gw_id * 90 if curr_gw_id > 0 else 38 * 90
players_df['minutes_confidence'] = (players_df['minutes'] / conf_denominator).clip(upper=1.0)

# Apply confidence multiplier before normalization — dampens small-sample outliers.
# All are per-game averages that produce inflated values for players with very few appearances.
#
# ppg_last uses minutes_last (from GW0) with a full-season denominator (38 * 90)
# since it reflects last season's data — current season's minutes are irrelevant for it.
ppg_last_confidence = (players_df['minutes_last'] / (38 * 90)).clip(upper=1.0)
players_df['ppg_last'] = players_df['ppg_last'] * ppg_last_confidence

# All other columns use current-season minutes_confidence.
for col in ['clean_sheets_per_90', 'saves_per_90', 'defensive_contribution', 'points_per_game', 'form']:
  players_df[col] = players_df[col] * players_df['minutes_confidence']

# FPL stores cost * 10 (e.g. 60 = £6.0m). Derive actual cost and value metric.
players_df['cost'] = players_df['now_cost'] / 10
players_df['points_per_euro'] = players_df['total_points'] / players_df['cost']

# Build FDR (Fixture Difficulty Rating) per team for the upcoming GW.
# Stack home and away sides, then map each team's difficulty onto players.
home = fixtures_df[['team_h', 'team_h_difficulty']].rename(columns={'team_h': 'team_id', 'team_h_difficulty': 'fdr'})
away = fixtures_df[['team_a', 'team_a_difficulty']].rename(columns={'team_a': 'team_id', 'team_a_difficulty': 'fdr'})
fdr_df = pd.concat([home, away])
players_df['fdr'] = players_df['team'].map(dict(zip(fdr_df['team_id'], fdr_df['fdr'])))

# Rebuild as a contiguous copy to resolve pandas memory fragmentation warning.
players_df = players_df.copy()

# Min-max normalize all scoring columns to 0-1 so weights are meaningful across different scales.
# FDR and saves_per_90 are inverted: lower raw value = better for the player.
# Guard: if all values are identical (e.g. form = 0.0 pre-season), denominator = 1 → everyone scores 0.
for col in PLAYERS_NORMALIZATION_COLUMNS:
  col_min = players_df[col].min()
  col_max = players_df[col].max()
  denominator = col_max - col_min if col_max != col_min else 1
  if col in ['fdr', 'saves_per_90']:
    players_df[f'{col}_norm'] = 1 - ((players_df[col] - col_min) / denominator)
  else:
    players_df[f'{col}_norm'] = (players_df[col] - col_min) / denominator


### Scoring

Applies position-specific weights to normalized columns. Each position uses different factors backed by correlation analysis (`explore.py`).

In [ ]:
# Scoring

# Resolve season-aware blend weights for the consistency signal.
# Returns (ppg_last_w, ppg_curr_w, form_curr_w) based on how many GWs have been played.
# All three sum to 1.0, and the combined FORM_AND_PPG budget is 0.20 of the total score.
[ppg_last_w, ppg_curr_w, form_curr_w] = get_season_weights(curr_gw_id)

FORM_AND_PPG_WEIGHTS = {
  'ppg_last': 0.20 * ppg_last_w,        # last season's ppg from GW0 snapshot
  'points_per_game': 0.20 * ppg_curr_w, # current season running average
  'form': 0.20 * form_curr_w             # rolling recent GW average
}

# Merge position-specific weights with the shared consistency signal.
# Each merged dict sums to 1.0: position weights (0.80) + FORM_AND_PPG (0.20).
gpk_weights = {**GKP_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
def_weights = {**DEF_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
mid_weights = {**MID_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
fwd_weights = {**FWD_WEIGHTS, **FORM_AND_PPG_WEIGHTS}

# Compute next_gw_score per position using a weighted sum of normalized columns.
mask = players_df['position'] == 'GKP'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in gpk_weights.items()])
mask = players_df['position'] == 'DEF'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in def_weights.items()])
mask = players_df['position'] == 'MID'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in mid_weights.items()])
mask = players_df['position'] == 'FWD'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in fwd_weights.items()])


### Recommendations

Top 15 players per position ranked by `next_gw_score`.

In [ ]:
# Visualizations
 
rec_cols = ['full_name', 'team_name', 'position', 'fdr', 'minutes', 'cost', 'total_points', 'points_per_euro', 'next_gw_score']
display_format = { 
  'next_gw_score': '{:.2f}',
   'cost': '£{:.1f}',
   'position_rank': '{:.0f}',
   'clean_sheets_per_90': '{:.2f}',
   'saves_per_90': '{:.2f}',
   'points_per_euro': '{:.2f}',
   'differential_score': '{:.2f}',
   'selected_by_percent': '{:.2f}',
}
recommendations = players_df.sort_values(by=['position', 'next_gw_score'], ascending=False).groupby('position').head(15)[rec_cols]

for _, pos in POSITION_MASTER.items():
  temp_df = recommendations[recommendations['position'] == pos]
  display(
    temp_df.sort_values(by='next_gw_score', ascending=False)
    .style
    .hide(axis='index')
    .set_caption(pos)
    .background_gradient(subset=['next_gw_score'], cmap='Greens')
    .format({**display_format})
  )

### Differential Picks

Low-ownership players with strong scoring potential. `differential_score = next_gw_score × (1 - ownership_norm)`, where ownership is normalized per position.

In [ ]:
# Differential Player (By Position)

col = 'selected_by_percent'
diff_cols = rec_cols + ['selected_by_percent', 'differential_score']
for _, pos in POSITION_MASTER.items():
  temp_df = players_df[players_df['position'] == pos]
  mask = players_df['position'] == pos
  if temp_df.empty:
    continue
  col_min = temp_df[col].min()
  col_max = temp_df[col].max()
  denominator = col_max - col_min if col_max != col_min else 1
  players_df.loc[mask, 'selected_by_percent_norm'] = 1 - ((temp_df[col] - col_min) / denominator)
  
players_df['differential_score'] = players_df['next_gw_score'] * players_df['selected_by_percent_norm']

differentials = players_df.sort_values(by=['position', 'differential_score'], ascending=False).groupby('position').head(15)[diff_cols]

for _, pos in POSITION_MASTER.items():
  temp_df = differentials[differentials['position'] == pos]
  if temp_df.empty:
    continue
  display(
    temp_df[diff_cols]
    .sort_values(by='differential_score', ascending=False)
    .style
    .hide(axis='index')
    .set_caption(f"{pos} ({len(players_df[players_df['position'] == pos])})")
    .background_gradient(subset=['differential_score'], cmap='Greens')
    .format({**display_format})
  )


### Watchlist

Track specific players by their `code` (permanent FPL ID). Shows position rank alongside scoring and differential data.

In [ ]:
# Watchlist ranking

watchlist_cols = ['code'] + rec_cols + ['position_rank', 'differential_score', 'selected_by_percent']
# add player codes (permanent FPL code, not id)
watchlist_players = [] 
my_team_players = [] 
# add team_code values to show all players from a team
watchlist_teams = []    

def highlight(row):
  # return a list of CSS strings, one per column
  if row['code'] in my_team_players: 
    return(['background-color: #8b3a3a'] * len(row))
  if row['code'] in watchlist_players: 
    return(['background-color: #3d1515'] * len(row))
  else:
    return(['background-color: #0d2137'] * len(row))

players_df['position_rank'] = players_df.groupby('position')['next_gw_score'].rank(ascending=False, method='min')  

mask = (players_df['code'].isin(watchlist_players)
        | players_df['code'].isin(my_team_players)
        | players_df['team_code'].isin(watchlist_teams))
filtered_players = players_df[mask]

if len(filtered_players):
  display(HTML("""
    <div style='margin-bottom:12px; font-size:13px;'>
      <span style='background-color:#8b3a3a; padding:3px 10px; border-radius:3px; margin-right:8px;'>&#9632; My team player</span>
      <span style='background-color:#3d1515; padding:3px 10px; border-radius:3px; margin-right:8px;'>&#9632; Watchlist player</span>
      <span style='background-color:#0d2137; padding:3px 10px; border-radius:3px;'>&#9632; Team player</span>
    </div>
  """))

for _, pos in POSITION_MASTER.items():
  temp_df = filtered_players[filtered_players['position'] == pos]
  if temp_df.empty:
    continue

  display(
    temp_df[watchlist_cols]
    .sort_values(by='position_rank')
    .style
    .apply(highlight, axis=1)
    .hide(axis='index')
    .hide(subset=['code'], axis='columns') 
    .set_caption(f"{pos} ({len(players_df[players_df['position'] == pos])})")
    .background_gradient(subset=['next_gw_score'], cmap='Greens')
    .format({**display_format})
  )